In [1]:
print("hello")

hello


In [2]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
import time

from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from imblearn.pipeline import Pipeline as ImbPipeline

from src.features import build_feature_creator, build_column_transformer

train = pd.read_csv("../data/processed/train.csv")
X_train = train.drop(columns="Class")
y_train = train["Class"]

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

In [3]:
xgb_pipeline = ImbPipeline([
    ("features", build_feature_creator()),
    ("prep", build_column_transformer()),
    ("model", XGBClassifier(
        scale_pos_weight=scale_pos_weight,
        eval_metric="aucpr",
        random_state=42,
        n_jobs=-1,
    )),
])

In [4]:
param_distributions = {
    "model__n_estimators": [100, 200, 300, 400, 500],
    "model__max_depth": [3, 4, 5, 6, 7, 8],
    "model__learning_rate": [0.01, 0.02, 0.05, 0.1, 0.2],
    "model__subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
    "model__colsample_bytree": [0.6, 0.7, 0.8, 0.9, 1.0],
    "model__min_child_weight": [1, 3, 5, 7],
}

In [5]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

search = RandomizedSearchCV(
    estimator=xgb_pipeline,
    param_distributions=param_distributions,
    n_iter=40,                       # nombre de combinaisons testées
    scoring="average_precision",     # toujours la PR-AUC
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=1,                       # affiche la progression
)

start = time.time()
search.fit(X_train, y_train)
print(f"Durée totale : {time.time() - start:.0f}s")

Fitting 5 folds for each of 40 candidates, totalling 200 fits
Durée totale : 892s


In [6]:
print("Meilleurs paramètres :")
for k, v in search.best_params_.items():
    print(f"  {k}: {v}")

print(f"\nMeilleure PR-AUC (CV) : {search.best_score_:.4f}")

Meilleurs paramètres :
  model__subsample: 0.7
  model__n_estimators: 300
  model__min_child_weight: 1
  model__max_depth: 8
  model__learning_rate: 0.1
  model__colsample_bytree: 0.7

Meilleure PR-AUC (CV) : 0.8550


In [7]:
print("Meilleurs paramètres :")
for k, v in search.best_params_.items():
    print(f"  {k}: {v}")

Meilleurs paramètres :
  model__subsample: 0.7
  model__n_estimators: 300
  model__min_child_weight: 1
  model__max_depth: 8
  model__learning_rate: 0.1
  model__colsample_bytree: 0.7


In [8]:
results_df = pd.DataFrame(search.cv_results_)
top5 = results_df.sort_values("rank_test_score").head(5)
top5[["mean_test_score", "std_test_score", "params"]]

,mean_test_score,std_test_score,params
24,0.855021,0.020608,"{'model__subsample': 0.7, 'model__n_estimators..."
17,0.854048,0.020127,"{'model__subsample': 0.8, 'model__n_estimators..."
38,0.852992,0.017266,"{'model__subsample': 0.8, 'model__n_estimators..."
16,0.851801,0.018808,"{'model__subsample': 0.7, 'model__n_estimators..."
2,0.851412,0.020133,"{'model__subsample': 0.6, 'model__n_estimators..."


In [9]:
best_xgb_pipeline = search.best_estimator_

import joblib
import os

os.makedirs("../models", exist_ok=True)
joblib.dump(best_xgb_pipeline, "../models/model.joblib")
print("Modèle sauvegardé dans models/model.joblib")

Modèle sauvegardé dans models/model.joblib
